In [ ]:
%%bash
mkdir -p ref
cd ref
wget -O GRCh38_chr10.fa.gz ftp://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz
gunzip -c GRCh38_chr10.fa.gz > GRCh38_chr10.fa
minimap2 -d GRCh38_chr10.mmi GRCh38_chr10.fa

--2025-11-01 16:40:12--  ftp://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz
           => ‘GRCh38_chr10.fa.gz’
Resolving hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)... 128.114.119.163
Connecting to hgdownload.soe.ucsc.edu (hgdownload.soe.ucsc.edu)|128.114.119.163|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /goldenPath/hg38/chromosomes ... done.
==> SIZE chr10.fa.gz ... 43157332
==> PASV ... done.    ==> RETR chr10.fa.gz ... done.
Length: 43157332 (41M) (unauthoritative)

     0K .......... .......... .......... .......... ..........  0%  824K 51s
    50K .......... .......... .......... .......... ..........  0% 1.63M 38s
   100K .......... .......... .......... .......... ..........  0% 5.28M 28s
   150K .......... .......... .......... .......... ..........  0% 4.63M 23s
   200K .......... .......... .......... .......... ..........  0% 2.65M 22s
   250K .......... ........

In [ ]:
%%bash
# Align short-read Illumina sample to chromosome 10



# Step 1: Uncompress Illumina sample and align reads and save SAM file
bzip2 -dc samples/illumina.fq.bz2 | minimap2 -ax sr GRCh38_chr10.mmi - -o illumina_chr10.sam

# Step 2: Convert SAM to sorted BAM
samtools sort illumina_chr10.sam -o illumina_chr10.sorted.bam

# Step 3: Create BAM index (.bai)
samtools index illumina_chr10.sorted.bam

# Same thing for PacBio
bzip2 -dc samples/pacbio.fq.bz2 | minimap2 -ax map-pb GRCh38_chr10.mmi - -o pacbio_chr10.sam
samtools sort pacbio_chr10.sam -o pacbio_chr10.sorted.bam
samtools index pacbio_chr10.sorted.bam

[WARNING] Indexing parameters (-k, -w or -H) overridden by parameters used in the prebuilt index.
[M::main::0.735*0.91] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::0.735*0.91] mid_occ = 1000
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::0.873*0.92] distinct minimizers: 16061920 (79.91% are singletons); average occurrences: 1.563; average spacing: 5.329; total length: 133797422
[M::worker_pipeline::11.438*2.44] mapped 309505 sequences
[M::main] Version: 2.30-r1287
[M::main] CMD: minimap2 -ax sr -o illumina_chr10.sam GRCh38_chr10.mmi -
[M::main] Real time: 11.456 sec; CPU: 27.970 sec; Peak RSS: 0.939 GB
[WARNING] Indexing parameters (-k, -w or -H) overridden by parameters used in the prebuilt index.
[M::main::0.619*0.99] loaded/built the index for 1 target sequence(s)
[M::mm_mapopt_update::0.792*1.00] mid_occ = 178
[M::mm_idx_stat] kmer size: 15; skip: 10; is_hpc: 0; #seq: 1
[M::mm_idx_stat::0.910*1.00] distinct minimizers: 160619

In [3]:
%%bash
# Call variants with FreeBayes
freebayes -f ref/GRCh38_chr10.fa illumina_chr10.sorted.bam > illumina_chr10.vcf
freebayes -f ref/GRCh38_chr10.fa pacbio_chr10.sorted.bam > pacbio_chr10.vcf

In [ ]:

# haptreex -v illumina_chr10.vcf -d illumina_chr10.sorted.bam -o illumina_chr10_phased.vcf

Reading chr10 took 0.00117s
CError: dlopen(libhts.dylib, 0x000A): tried: 'libhts.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibhts.dylib' (no such file), '/Users/michellerosenthal/bin/haptreex-mac/libhts.dylib' (no such file), '/Users/michellerosenthal/bin/haptreex-mac/libhts.dylib' (no such file), '/usr/lib/libhts.dylib' (no such file, not in dyld cache), 'libhts.dylib' (no such file), '/usr/local/lib/libhts.dylib' (no such file), '/usr/lib/libhts.dylib' (no such file, not in dyld cache)

raised from: dlopen
/Users/inumanag/.seq/lib/seq/stdlib/core/dlopen.seq:16:9

backtrace:
  [0x102e060a0] (?)
  [0x102e1282f] (?)
  [0x102e0ce04] (?)
  [0x102e0c7d5] (?)
  [0x102e2b2b1] .omp_outlined..860 (+0x481)
  [0x10b4ce82b] _ZL17__kmp_invoke_taskiP8kmp_taskP12kmp_taskdata (+0x2bb)
  [0x10b4cf5ef] __kmp_execute_tasks_32 (+0x35f)
  [0x10b4cedd3] __kmpc_omp_taskwait (+0x143)
  [0x102e0afd7] (?)
  [0x102e255ed] main (+0xa3d)


HapTree-X v2.0
Ploidy: 2
Loading VCF file illumina_chr10.vcf...
1027 SNPs in VCF file
Running DNA/RNA phasing without DASE (no gene data provided)


bash: line 1: 50933 Abort trap: 6           haptreex -v illumina_chr10.vcf -d illumina_chr10.sorted.bam -o illumina_chr10_phased.vcf


CalledProcessError: Command 'b'haptreex -v illumina_chr10.vcf -d illumina_chr10.sorted.bam -o illumina_chr10_phased.vcf\n'' returned non-zero exit status 134.